# 73. 기존 CN7 가스에 대한 고정 RG3 모델 전이 진단
71번모델·피처·임계값규칙을바꾸지않고기존713개CN7 RH에적용한다. 모델은RG3620개만학습한다. 이미분석한CN7가스를새독립검증이라주장하지않는다. RH만사용하며전체CN7가스13개(LH포함)와혼동하지않는다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/71_rg3_to_cn7_extra_blind_predictions.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
s=s.replace('meta=meta[meta.known.isna()]','meta=meta[meta.known.eq(True)]').replace('assert len(meta)==1276','assert len(meta)==713').replace('cn7_transfer1276_candidate_input_audit.csv','cn7_known713_input_audit.csv')
s=s.split("path=tables/'cn7_transfer1276_predictions_SEALED.csv'")[0]
exec(s)
raw=pd.read_csv(root/'data/original_split/moldset_labeled_cn7.csv')
raw=raw[raw.PART_NAME.str.contains('RH',na=False)].copy();raw.TimeStamp=pd.to_datetime(raw.TimeStamp)
g=pred.merge(raw[['EQUIP_CD','PART_NAME','TimeStamp','PassOrFail','Reason']],on=['EQUIP_CD','PART_NAME','TimeStamp'],validate='one_to_one')
gas=g.PassOrFail.eq(1)&g.Reason.fillna('').str.contains('가스');normal=g.PassOrFail.eq(0)
g['eligible']=gas|normal;g['actual_gas']=gas.astype(int)
rows=[]
for name in ['reference','primary','secondary']:
 q=g[g.eligible];tn,fp,fn,tp=confusion_matrix(q.actual_gas,q[name+'_prediction'],labels=[0,1]).ravel()
 rows.append(dict(model=name,normal=int(normal.sum()),gas=int(gas.sum()),other_defects=int((~g.eligible).sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn)))
m=pd.DataFrame(rows)
g.to_csv(tables/'cn7_known713_rg3_transfer_predictions.csv',index=False,encoding='utf-8-sig')
m.to_csv(tables/'cn7_known713_rg3_transfer_metrics.csv',index=False,encoding='utf-8-sig')
print('KNOWN CN7 development transfer, NOT new blind validation')
print(m.to_string(index=False))
print('Gas cases',g[gas][['TimeStamp','primary_prediction','reference_prediction','secondary_prediction']].to_string(index=False))
